# 06 — Dataset dan DataLoader Citra Awan

Notebook ini membangun objek `Dataset` dan `DataLoader` PyTorch untuk klasifikasi
jenis awan menggunakan hasil dari tahap split, preprocessing, dan augmentasi.

Notebook ini tidak melakukan pembagian dataset, pembersihan citra, atau pembuatan
augmentasi baru. Seluruh informasi dataset dibaca dari metadata yang sudah disimpan
di `dataset/processed/`.

Tujuan notebook:

1. Membaca indeks dataset hasil notebook 05.
2. Membaca pemetaan kelas yang sudah ditetapkan.
3. Memuat pipeline transform training dan evaluasi.
4. Membentuk custom PyTorch Dataset.
5. Membentuk DataLoader train, validation, dan test.
6. Memastikan dimensi tensor dan label sesuai.
7. Memeriksa distribusi kelas pada DataLoader.
8. Memvisualisasikan satu batch training.
9. Menyimpan konfigurasi DataLoader.
10. Menyiapkan input untuk tahap training model.

Data test hanya dibentuk sebagai DataLoader. Data test tidak dianalisis atau digunakan
untuk pemilihan model dan hyperparameter.

## Mengimpor Library

Library yang digunakan terbatas pada pengelolaan metadata, pembacaan citra,
transformasi, visualisasi, dan pembentukan DataLoader PyTorch.

In [1]:
from pathlib import Path
import json
import math
import random
import sys

try:
    import albumentations as A
    import cv2
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import torch

    from IPython.display import display
    from torch.utils.data import (
        DataLoader,
        Dataset,
        RandomSampler,
        get_worker_info,
    )

    # Import ini juga memastikan ToTensorV2 terdaftar
    # ketika pipeline Albumentations dimuat dari JSON.
    from albumentations.pytorch import ToTensorV2

except ImportError as exc:
    raise ImportError(
        "Dependency DataLoader belum lengkap.\n"
        "Aktifkan .venv, kemudian jalankan melalui terminal VS Code:\n\n"
        r".\.venv\Scripts\python.exe -m pip install -r requirements.txt"
    ) from exc


pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

cv2.setNumThreads(0)


print(f"Python          : {sys.version.split()[0]}")
print(f"NumPy           : {np.__version__}")
print(f"Pandas          : {pd.__version__}")
print(f"PyTorch         : {torch.__version__}")
print(f"OpenCV          : {cv2.__version__}")
print(f"Albumentations  : {A.__version__}")
print(f"CUDA tersedia   : {torch.cuda.is_available()}")

Python          : 3.11.9
NumPy           : 2.4.6
Pandas          : 3.0.5
PyTorch         : 2.13.0+cpu
OpenCV          : 5.0.0
Albumentations  : 2.0.8
CUDA tersedia   : False


## Menentukan Lokasi Project

Lokasi project ditentukan secara dinamis agar notebook dapat dijalankan dari folder
utama project maupun folder `notebooks`.

Notebook tidak menggunakan path absolut Docker `/app`.

In [2]:
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR


DATASET_DIR = PROJECT_DIR / "dataset"
RAW_DIR = DATASET_DIR / "raw"
SOURCE_DIR = DATASET_DIR / "source"
PROCESSED_DIR = DATASET_DIR / "processed"

MODELS_DIR = PROJECT_DIR / "models"
LOGS_DIR = PROJECT_DIR / "logs"

SOURCE_TRAIN_DIR = SOURCE_DIR / "train"
SOURCE_VAL_DIR = SOURCE_DIR / "val"
SOURCE_TEST_DIR = SOURCE_DIR / "test"


DATASET_INDEX_PATH = (
    PROCESSED_DIR
    / "preprocessing_dataset_index.csv"
)

CLASS_MAPPING_PATH = (
    PROCESSED_DIR
    / "class_mapping.json"
)

PREPROCESSING_CONFIG_PATH = (
    PROCESSED_DIR
    / "preprocessing_config.json"
)

TRAIN_TRANSFORM_PATH = (
    PROCESSED_DIR
    / "train_transform.json"
)

EVAL_TRANSFORM_PATH = (
    PROCESSED_DIR
    / "eval_transform.json"
)

DATALOADER_SUMMARY_PATH = (
    PROCESSED_DIR
    / "dataloader_summary.csv"
)

DATALOADER_CONFIG_PATH = (
    PROCESSED_DIR
    / "dataloader_config.json"
)


path_table = pd.DataFrame({
    "Nama": [
        "PROJECT_DIR",
        "SOURCE_DIR",
        "PROCESSED_DIR",
        "DATASET_INDEX_PATH",
        "CLASS_MAPPING_PATH",
        "PREPROCESSING_CONFIG_PATH",
        "TRAIN_TRANSFORM_PATH",
        "EVAL_TRANSFORM_PATH",
    ],
    "Path": [
        PROJECT_DIR,
        SOURCE_DIR,
        PROCESSED_DIR,
        DATASET_INDEX_PATH,
        CLASS_MAPPING_PATH,
        PREPROCESSING_CONFIG_PATH,
        TRAIN_TRANSFORM_PATH,
        EVAL_TRANSFORM_PATH,
    ],
})

path_table["Ada"] = path_table["Path"].map(Path.exists)

display(path_table)

,Nama,Path,Ada
0,PROJECT_DIR,C:\Users\jardm\Documents\n8n-logsiswaparalayang\cloud-classification,True
1,SOURCE_DIR,C:\Users\jardm\Documents\n8n-logsiswaparalayang\cloud-classification\dataset\source,True
2,PROCESSED_DIR,C:\Users\jardm\Documents\n8n-logsiswaparalayang\cloud-classification\dataset\processed,True
3,DATASET_INDEX_PATH,C:\Users\jardm\Documents\n8n-logsiswaparalayang\cloud-classification\dataset\processed\preprocessing_dataset_index.csv,True
4,CLASS_MAPPING_PATH,C:\Users\jardm\Documents\n8n-logsiswaparalayang\cloud-classification\dataset\processed\class_mapping.json,True
5,PREPROCESSING_CONFIG_PATH,C:\Users\jardm\Documents\n8n-logsiswaparalayang\cloud-classification\dataset\processed\preprocessing_config.json,True
6,TRAIN_TRANSFORM_PATH,C:\Users\jardm\Documents\n8n-logsiswaparalayang\cloud-classification\dataset\processed\train_transform.json,True
7,EVAL_TRANSFORM_PATH,C:\Users\jardm\Documents\n8n-logsiswaparalayang\cloud-classification\dataset\processed\eval_transform.json,True


## Konfigurasi DataLoader

`NUM_WORKERS` menggunakan nilai awal `0` karena notebook dijalankan langsung melalui
VS Code pada Windows. Nilai tersebut menghindari masalah multiprocessing pada kernel
Jupyter.

Setelah pipeline stabil, nilai `NUM_WORKERS` dapat diuji menjadi `2` atau `4`.
Peningkatan nilai harus mempertimbangkan jumlah inti CPU, RAM, dan kecepatan media
penyimpanan.

`drop_last=False` digunakan agar seluruh citra tetap dipakai, termasuk batch terakhir
yang jumlahnya lebih kecil daripada ukuran batch.

In [3]:
SEED = 42
BATCH_SIZE = 32

# Nilai paling stabil untuk VS Code/Jupyter pada Windows.
NUM_WORKERS = 0

PIN_MEMORY = torch.cuda.is_available()
PERSISTENT_WORKERS = NUM_WORKERS > 0
DROP_LAST = False

SPLIT_ORDER = [
    "train",
    "val",
    "test",
]


def seed_everything(seed: int) -> None:
    """
    Menetapkan seed utama agar eksperimen lebih mudah direproduksi.
    """
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)


configuration_table = pd.DataFrame({
    "Parameter": [
        "SEED",
        "BATCH_SIZE",
        "NUM_WORKERS",
        "PIN_MEMORY",
        "PERSISTENT_WORKERS",
        "DROP_LAST",
    ],
    "Nilai": [
        SEED,
        BATCH_SIZE,
        NUM_WORKERS,
        PIN_MEMORY,
        PERSISTENT_WORKERS,
        DROP_LAST,
    ],
})

display(configuration_table)

,Parameter,Nilai
0,SEED,42
1,BATCH_SIZE,32
2,NUM_WORKERS,0
3,PIN_MEMORY,False
4,PERSISTENT_WORKERS,False
5,DROP_LAST,False


## Memeriksa Prasyarat

Notebook dihentikan apabila folder source, metadata preprocessing, pemetaan kelas,
atau pipeline transform belum tersedia.

Pemeriksaan ini tidak memindai ulang dataset mentah dan tidak menghitung ulang hash.
Integritas dan kebocoran dataset sudah diperiksa pada notebook 04.

In [4]:
required_directories = [
    PROJECT_DIR,
    DATASET_DIR,
    SOURCE_DIR,
    SOURCE_TRAIN_DIR,
    SOURCE_VAL_DIR,
    SOURCE_TEST_DIR,
    PROCESSED_DIR,
]


required_files = [
    DATASET_INDEX_PATH,
    CLASS_MAPPING_PATH,
    PREPROCESSING_CONFIG_PATH,
    TRAIN_TRANSFORM_PATH,
    EVAL_TRANSFORM_PATH,
]


missing_directories = [
    path
    for path in required_directories
    if not path.is_dir()
]

missing_files = [
    path
    for path in required_files
    if not path.is_file()
]


if missing_directories:
    formatted_paths = "\n".join(
        f"- {path}"
        for path in missing_directories
    )

    raise FileNotFoundError(
        "Folder prasyarat belum tersedia:\n"
        f"{formatted_paths}"
    )


if missing_files:
    formatted_paths = "\n".join(
        f"- {path}"
        for path in missing_files
    )

    raise FileNotFoundError(
        "Metadata atau transform tahap 05 belum tersedia:\n"
        f"{formatted_paths}\n\n"
        "Jalankan kembali 05_preprocessing_augmentasi.ipynb "
        "sebelum menjalankan notebook ini."
    )


print("Seluruh folder, metadata, dan transform tersedia.")

Seluruh folder, metadata, dan transform tersedia.
